In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 07 consolidation — Understand why the same points give different complexes

Read `sessions/07_session.md` first. This is an executed reference, not an assessment of you.

**Evidence:** Real Iris point sample; exact equilateral-triangle scale derivation.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT/'data/raw/iris.csv').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Start Jupyter in the extracted listening_to_shape_lab folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))


## Step 1

Calculate the two geometric scales before asking any library for a persistence diagram.

Before running: write your expected result and name one way this check could fail.

In [3]:
edge_rips = 1.
cech_pair_radius = .5
cech_triple_radius = 1/np.sqrt(3)
print('Rips edge scale:', edge_rips)
print('Cech radii:', cech_pair_radius, cech_triple_radius)
assert cech_triple_radius > cech_pair_radius

Rips edge scale: 1.0
Cech radii: 0.5 0.5773502691896258


## Step 2

Build a filtration from actual measurements. The stored tuple records each simplex and its entry value.

Before running: write your expected result and name one way this check could fail.

In [4]:
from shape_lab.data import load_iris_data
from shape_lab.persistence import rips_filtration
from shape_lab.algebra import betti_numbers
frame = load_iris_data().iloc[::15]
X = frame[[c for c in frame if c.endswith('_cm')]].to_numpy()
filtration = rips_filtration(X, max_homology=1)
values = dict(filtration)
from itertools import combinations
for simplex, value in filtration:
    if len(simplex)>1:
        assert all(values[face] <= value for face in combinations(simplex, len(simplex)-1))
print('records:', len(X), 'stored simplices:', len(filtration))

records: 10 stored simplices: 175


## Step 3

Compare several nested complexes. Their simplex counts are monotone; their Betti numbers need not be.

Before running: write your expected result and name one way this check could fail.

In [5]:
scales = [0., .5, 1., 2., 5., 10.]
rows = []
for scale in scales:
    K = [s for s,t in filtration if t <= scale]
    rows.append([scale, len(K), *betti_numbers(K, 1)])
import pandas as pd
display(pd.DataFrame(rows, columns=['edge threshold (cm)', 'simplices', 'beta0', 'beta1']))
assert np.all(np.diff([row[1] for row in rows]) >= 0)
result = {'sample_ids': frame.sample_id.astype(int).tolist(), 'filtration_rows': rows,
          'scale_convention': 'maximum pairwise Euclidean edge length in selected cm features'}

,edge threshold (cm),simplices,beta0,beta1
0,0.0,10,10,0
1,0.5,11,9,0
2,1.0,16,6,0
3,2.0,27,2,0
4,5.0,101,1,0
5,10.0,175,1,0


## Keep execution evidence separate from learning evidence

The following saves this reference calculation. Write your own explanation in `my_work/`; no learner score is assigned.

In [6]:
result['stage'] = 7
result['execution_role'] = 'reference_consolidation'
result['data_role'] = 'Real Iris point sample; exact equilateral-triangle scale derivation.'
output = ROOT/'reports/consolidation_v4/07.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Saved reference result:', output.relative_to(ROOT))


Saved reference result: reports/consolidation_v4/07.json
